# 01 -- Environment check and dataset audit

Run this notebook **once per dataset**: first with `DATASET = "lavdf"`, then with `DATASET = "fakeavceleb"` (section 1.2).

It (1) checks hardware and packages and (2) audits the folder layout, then builds `manifest.json`: one row per clip with a label, a group id (`subject`) and a generation method.

The audit has **hard gates**. If labels or group ids cannot be parsed, the notebook stops, because silent parsing failures would turn the group-disjoint split in notebook 03 into a random split and inflate every result.

* **LAV-DF**: labels come from `metadata.json` (`n_fakes == 0` is real, otherwise fake); the group id is the fake's `original` source clip. LAV-DF has a single generation pipeline, so a held-out-method split is not possible (the audit prints this as an acknowledged limitation, not a failure).
* **FakeAVCeleb**: labels and methods come from `meta_data.csv`; the group id is the `idNNNNN` speaker folder.
* A FaceForensics++ branch is kept in the parser for completeness but was **not used**: the copy we obtained contained no audio streams.

## 1.1 Hardware and environment

In [ ]:
import sys, platform, subprocess

print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())

try:
    import torch
    print("torch  :", torch.__version__)
    print("CUDA available :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU    :", torch.cuda.get_device_name(0))
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"VRAM   : {vram:.1f} GB")
        if vram < 20:
            print("!! Less than 20 GB VRAM: reduce batch_size in notebook 03 if you run out of memory.")
    else:
        print("\n!! No CUDA. Feature extraction will be painfully slow on CPU.")
except ImportError:
    print("torch not installed -- run the install cell below.")

In [ ]:
# Dependencies: see ../requirements.txt  (pip install -r requirements.txt)
# Audio is read from the .mp4 files through torchaudio's FFmpeg backend, so a system
# FFmpeg (shared libraries) must be installed, e.g.  sudo apt install ffmpeg
print("Install dependencies once, outside the notebook.")

## 1.2 Point at your data

Set `DATASET` below. Raw data are expected under `../data/raw/` (see `docs/DATA.md`); nothing in `data/` is committed to the repository.

In [ ]:
from pathlib import Path

DATASET = "lavdf"        # "lavdf"  = pass 1, primary training corpus
                         # "fakeavceleb" = pass 2, cross-dataset test corpus
DATA_DIR = Path("../data")
PATHS = {
    "lavdf":       (DATA_DIR / "raw" / "LAV-DF",           DATA_DIR / "feats" / "lavdf"),
    "fakeavceleb": (DATA_DIR / "raw" / "FakeAVCeleb_v1.2", DATA_DIR / "feats" / "fakeavceleb"),
}
RAW_ROOT, FEAT_ROOT = PATHS[DATASET]
RUNS_ROOT = DATA_DIR / "runs"            # checkpoints and logs (notebooks 03 and 04)

for p in (FEAT_ROOT, RUNS_ROOT):
    p.mkdir(parents=True, exist_ok=True)

assert RAW_ROOT.exists(), f"RAW_ROOT does not exist: {RAW_ROOT.resolve()} -- see docs/DATA.md"
print("Dataset  :", DATASET)
print("Data root:", RAW_ROOT.resolve())

## 1.3 What is actually in there?

In [ ]:
import collections

videos = sorted(RAW_ROOT.rglob("*.mp4"))
print(f"Found {len(videos)} .mp4 files\n")
assert len(videos) > 0, "No videos found. Check RAW_ROOT and that the zip extracted."

print("--- 15 sample paths (relative to root) ---")
for v in videos[:15]:
    print(" ", v.relative_to(RAW_ROOT))

print("\n--- directory depth distribution ---")
depths = collections.Counter(len(v.relative_to(RAW_ROOT).parts) for v in videos)
for d, n in sorted(depths.items()):
    print(f"  depth {d}: {n} files")

print("\n--- top-level entries ---")
for p in sorted(RAW_ROOT.iterdir())[:20]:
    print("  ", p.name, "(dir)" if p.is_dir() else "")

**Look at the output above before continuing.**

The *official* FaceForensics++ layout looks roughly like:

```
FaceForensics++/
  original_sequences/youtube/c23/videos/000.mp4
  manipulated_sequences/Deepfakes/c23/videos/000_003.mp4
  manipulated_sequences/Face2Face/c23/videos/000_003.mp4
  manipulated_sequences/FaceSwap/c23/videos/000_003.mp4
  manipulated_sequences/NeuralTextures/c23/videos/000_003.mp4
  ...
```

`original_sequences` vs `manipulated_sequences` gives you the **label**. The method folder (`Deepfakes`, `Face2Face`, `FaceSwap`, `NeuralTextures`, ...) gives you the **generation method**. The filename `<target>_<source>.mp4` (or `<id>.mp4` for reals) gives you the **subject** — the target identity is the one you must not leak between train and test. The `raw`/`c23`/`c40` folder gives the **compression level**.

If your download looks different — a subset of methods only, a different compression level kept, a Kaggle-style flattened mirror — you must adapt the parser in the next cell. The audit that follows will tell you whether you succeeded.

## 1.4 The parser — adapt this to YOUR layout

Auto-detects three possible layouts: FakeAVCeleb, FaceForensics++, or **LAV-DF** (metadata-driven — if `metadata.json` / `metadata_min.json` sits at the root of `RAW_ROOT`, that's LAV-DF, and ground truth comes from the JSON, not from regexing paths).

**Note on LAV-DF specifically:** it has no `id#####`-style identity folders and only one generation pipeline (not multiple methods like FF++/FakeAVCeleb). So:
- "subject" here is a **surrogate** — the fake's `original` source-video pointer (or the file itself, for reals) — grouped so a fake and its own real source never split across train/test. This is video-group-disjoint, **not** true speaker-disjoint** (the same person can appear as the source of multiple different `original` videos, and the metadata gives no speaker ID to catch that).
- **Method-disjoint splitting is not possible on LAV-DF alone** — there is nothing to hold out. The audit below acknowledges this rather than failing on it, but it means any generalisation claim from LAV-DF has to rely on the cross-dataset transfer to FF++, not a held-out-method test within LAV-DF.

In [ ]:
import re, json as _json
from pathlib import Path

# ============================================================================
# Auto-detects which dataset this RAW_ROOT is:
#   - LAV-DF        : metadata.json / metadata_min.json at the root (checked first --
#                      it's authoritative, no path-guessing needed)
#   - FaceForensics++: original_sequences / manipulated_sequences folders
#   - FakeAVCeleb    : RealVideo-RealAudio-style category folders
# Run this notebook once per dataset with the matching RAW_ROOT / FEAT_ROOT (cell 1.2).
# ============================================================================

DATASET_KIND = None   # set below; read by the audit's hard-gates cell

# any CSV under RAW_ROOT -- used by the LAV-DF metadata check below AND by
# the FakeAVCeleb meta_data.csv join further down
_csv_candidates = list(RAW_ROOT.rglob("*.csv"))

# ---- LAV-DF: metadata-driven (no regex needed) ----
_lavdf_meta_path = None
for _cand in ("metadata_min.json", "metadata.json"):
    if (RAW_ROOT / _cand).exists():
        _lavdf_meta_path = RAW_ROOT / _cand
        break

_LAVDF_INDEX = {}
if _lavdf_meta_path is not None:
    DATASET_KIND = "lavdf"
    print(f"Detected LAV-DF (found {_lavdf_meta_path.name}). Loading metadata...")
    _records = _json.loads(_lavdf_meta_path.read_text())
    for r in _records:
        _LAVDF_INDEX[r["file"]] = r
    print(f"  indexed {len(_LAVDF_INDEX)} metadata records")


def _parse_lavdf(rel_path: str):
    rec = _LAVDF_INDEX.get(rel_path)
    if rec is None:
        return None, None, None, None
    label = 0 if rec["n_fakes"] == 0 else 1
    category = "lavdf_real" if label == 0 else "lavdf_fake"
    # "subject" surrogate: group a fake with its own real source video so they
    # never split across train/test. Reals are their own group.
    subject = rec["original"] if rec["original"] else rec["file"]
    # single generation pipeline -- method-disjoint is not available on this
    # dataset, but we still tag modify_video/modify_audio for finer ablations later
    if label == 0:
        method = "real"
    else:
        tags = []
        if rec.get("modify_video"): tags.append("video")
        if rec.get("modify_audio"): tags.append("audio")
        method = "lavdf-" + "+".join(tags) if tags else "lavdf"
    return subject, category, method, label


# ---- FakeAVCeleb / FF++: path-based (fallback if no LAV-DF metadata found) ----

def _norm(s: str) -> str:
    return re.sub(r"[\s\-_]+", "", s.lower())

CATEGORY_KEYWORDS = {
    _norm("RealVideo-RealAudio"): 0,
    _norm("RealVideo-FakeAudio"): 1,
    _norm("FakeVideo-RealAudio"): 1,
    _norm("FakeVideo-FakeAudio"): 1,
    "real": 0, "realreal": 0, "fake": 1, "fakefake": 1,
}
METHOD_KEYWORDS = ["faceswap-wavtolip", "faceswapwavtolip", "faceswap-wav2lip",
                   "faceswapwav2lip", "faceswap", "fsgan",
                   "wavtolip", "wav2lip", "rtvc", "wav2clip"]  # 'wavtolip' is this
                   # mirror's actual spelling (confirmed from real sample paths) --
                   # kept 'wav2lip' too in case another mirror uses that spelling
SUBJECT_PATTERNS = [r"(id\d{4,6})", r"(id[_\-]?\d{3,6})", r"[/_\-](\d{4,6})[/_\-]"]
FFPP_METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures",
                "FaceShifter", "DeepFakeDetection"]


def _parse_ffpp(rel_path: str, low: str):
    if "original_sequences" in low:
        category, label = "original_sequences", 0
    elif "manipulated_sequences" in low:
        category, label = "manipulated_sequences", 1
    else:
        category, label = None, None
    method = "real" if label == 0 else None
    if label == 1:
        for meth in FFPP_METHODS:
            if f"/{meth.lower()}/" in low:
                method = meth
                break
    stem = Path(rel_path).stem
    # Standard FF++ methods (Deepfakes/Face2Face/FaceSwap/NeuralTextures):
    #   "000_003.mp4" (real: "000.mp4")           -- 3-digit target[_source] IDs
    # DeepFakeDetection (DFD) subset uses a DIFFERENT convention entirely:
    #   "01_02__exit_phone_room__YVGY8LOK.mp4"     -- 1-2 digit actor IDs,
    #                                                  then "__scene__hash"
    # Anchoring at the START and allowing 1-4 digits covers both: the leading
    # number is the on-screen (target) identity in both conventions.
    m = re.match(r"^(\d{1,4})", stem)
    subject = m.group(1) if m else None
    # DFD reuses short numeric actor IDs (01, 02, ...) that COLLIDE with the
    # standard methods' IDs (000, 001, ...) even though they refer to
    # different people. Prefix by method so "01" from DFD and "001" from
    # Deepfakes are never treated as the same subject.
    if subject is not None:
        subject = f"{method or 'real'}:{subject}"
    return subject, category, method, label


# ---- FakeAVCeleb: prefer meta_data.csv (authoritative) over filename regex ----
# Confirmed from real data: some fakes (e.g. FakeVideo-RealAudio via faceswap)
# carry NO method tag in the filename at all -- the CSV is the only reliable
# source for those. Columns confirmed from the actual file: source, target1,
# target2, method, category, type, race, gender, path, Unnamed: 9
FAVC_CSV_INDEX = {}
for _cpath in _csv_candidates:
    try:
        import pandas as pd
        _df = pd.read_csv(_cpath)
    except Exception:
        continue
    if {"source", "method", "type", "path"}.issubset(set(_df.columns)):
        print(f"Using FakeAVCeleb metadata CSV: {_cpath.name} ({len(_df)} rows)")
        for _, r in _df.iterrows():
            key = (str(r["source"]).strip(), str(r["path"]).strip())
            FAVC_CSV_INDEX[key] = {"method": str(r["method"]).strip().lower(),
                                    "type": str(r["type"]).strip()}
        break


def _parse_favc(low: str, raw: str = None):
    category, label = None, None
    for kw, lab in CATEGORY_KEYWORDS.items():
        if kw and kw in low.replace("-", "").replace("_", "").replace(" ", ""):
            category, label = kw, lab
            break
    subject = None
    for pat in SUBJECT_PATTERNS:
        m = re.search(pat, low)
        if m:
            subject = m.group(1)
            break

    # try the CSV first (authoritative -- covers files with no filename tag)
    if FAVC_CSV_INDEX and subject and raw:
        fname = Path(raw).name
        row = FAVC_CSV_INDEX.get((subject, fname))
        if row is not None:
            csv_cat = _norm(row["type"])
            csv_label = CATEGORY_KEYWORDS.get(csv_cat, label)  # fall back if type spelling differs
            return subject, csv_cat, row["method"], csv_label

    # fallback: filename-regex method (for any file not found in the CSV)
    method = "real" if label == 0 else None
    if label == 1:
        for meth in METHOD_KEYWORDS:
            if meth in low:
                method = meth
                break
    return subject, category, method, label


def parse_path(rel_path: str):
    """Return (subject, category, method, label) from a path relative to RAW_ROOT."""
    global DATASET_KIND
    raw = rel_path.replace("\\", "/")
    low = raw.lower()

    if _LAVDF_INDEX:
        return _parse_lavdf(raw)
    if "original_sequences" in low or "manipulated_sequences" in low:
        DATASET_KIND = DATASET_KIND or "ffpp"
        return _parse_ffpp(raw, low)
    subj, cat, meth, lab = _parse_favc(low, raw)
    if cat is not None:
        DATASET_KIND = DATASET_KIND or "fakeavceleb"
    return subj, cat, meth, lab


print("--- parser output on 15 samples (FULL paths, not truncated) ---")
for v in videos[:15]:
    rel = v.relative_to(RAW_ROOT).as_posix()
    print(f"  {parse_path(rel)}")
    print(f"    <-  {rel}")

# print AFTER the loop -- DATASET_KIND is only set as a side effect of the
# parse_path() calls above, so printing it before them (as an earlier version
# of this cell did) always showed a stale/wrong value
print(f"\nDATASET_KIND = {DATASET_KIND}")


## 1.5 THE AUDIT — this is the gate

These assertions fail if the parser can't do its job. **Do not proceed past a failure
by deleting the assertion.** Fix `parse_path` until it passes, or your splits are
worthless.

In [ ]:
import collections

rows = []
for v in videos:
    rel = v.relative_to(RAW_ROOT).as_posix()
    subj, cat, meth, lab = parse_path(rel)
    rows.append({"path": rel, "subject": subj, "category": cat,
                 "method": meth, "label": lab})

n = len(rows)
n_no_subject  = sum(r["subject"]  is None for r in rows)
n_no_category = sum(r["category"] is None for r in rows)
n_no_method   = sum(r["method"]   is None for r in rows)

print(f"Total videos      : {n}")
print(f"Missing subject   : {n_no_subject}   ({100*n_no_subject/n:.1f}%)")
print(f"Missing category  : {n_no_category}  ({100*n_no_category/n:.1f}%)")
print(f"Missing method    : {n_no_method}    ({100*n_no_method/n:.1f}%)")

print("\n--- label distribution ---")
for lab, c in sorted(collections.Counter(r["label"] for r in rows).items(),
                     key=lambda x: (x[0] is None, x[0])):
    name = {0: "REAL", 1: "FAKE"}.get(lab, "UNPARSED")
    print(f"  {name:9s}: {c:6d}  ({100*c/n:.1f}%)")

print("\n--- category distribution ---")
for cat, c in collections.Counter(r["category"] for r in rows).most_common():
    print(f"  {str(cat):22s}: {c:6d}")

print("\n--- method distribution ---")
for meth, c in collections.Counter(r["method"] for r in rows).most_common():
    print(f"  {str(meth):20s}: {c:6d}")

print("\n--- subjects ---")
subs = {r['subject'] for r in rows if r['subject']}
print(f"  unique subjects: {len(subs)}")
print(f"  examples: {sorted(subs)[:8]}")
# --- show actual paths that failed to parse, so a fix can be targeted ---
print("\n--- sample paths with NO category match (first 10) ---")
for r in [r for r in rows if r["category"] is None][:10]:
    print(" ", r["path"])

print("\n--- sample paths with NO subject match (first 10) ---")
for r in [r for r in rows if r["subject"] is None][:10]:
    print(" ", r["path"])

print("\n--- sample FAKE paths with NO method match (first 10) ---")
for r in [r for r in rows if r["label"] == 1 and r["method"] is None][:10]:
    print(" ", r["path"])


In [ ]:
# ---------------- HARD GATES ----------------
errs = []
warns = []

# small tolerance (not zero): a handful of stray files -- demo folders, junk
# samples that ship alongside some mirrors -- shouldn't hard-fail the whole
# audit. A layout that's genuinely wrong will blow way past this threshold.
CATEGORY_TOLERANCE = max(10, int(0.01 * n))
if n_no_category > CATEGORY_TOLERANCE:
    errs.append(f"{n_no_category} files have no parseable CATEGORY -> no label "
                f"(tolerance was {CATEGORY_TOLERANCE}). Your layout lacks the "
                "four Real/Fake category folders, or they are named differently. "
                "Fix CATEGORIES / parse_path.")
elif n_no_category > 0:
    warns.append(f"{n_no_category} files have no parseable CATEGORY and will be "
                 "DROPPED from the manifest (see the unparsed-path samples above -- "
                 "confirm they are junk/demo files, not real data you are silently "
                 "losing).")

if n_no_subject > 0.02 * n:
    errs.append(f"{n_no_subject} files have no parseable SUBJECT id. "
                "Subject-disjoint splitting is IMPOSSIBLE without this -- identities "
                "will leak across train/test and your scores will be inflated. "
                "Fix the subject regex in parse_path.")

fake_rows = [r for r in rows if r["label"] == 1]
n_fake_nometh = sum(r["method"] is None for r in fake_rows)
if fake_rows and n_fake_nometh > 0.05 * len(fake_rows):
    errs.append(f"{n_fake_nometh}/{len(fake_rows)} FAKE files have no parseable METHOD. "
                "Method-disjoint splitting is IMPOSSIBLE without this -- the model can "
                "memorise the generator it is tested on. Fix METHODS / parse_path.")

if len(subs) < 20:
    errs.append(f"Only {len(subs)} unique subjects found. Too few for a meaningful "
                "subject-disjoint split. Is this a small Kaggle subset rather than the "
                "full corpus?")

n_methods = len({r['method'] for r in fake_rows if r['method']})
if n_methods < 2:
    if DATASET_KIND == "lavdf":
        # LAV-DF genuinely only has one generation pipeline -- this is a real,
        # documented property of the dataset, not a parsing failure. Acknowledge
        # it instead of failing, but make it loud: the paper must not claim
        # method-disjoint generalisation from LAV-DF alone.
        warns.append(
            "LAV-DF has only ONE generation pipeline -- method-disjoint splitting "
            "is not possible on this corpus by design (not a parser bug). Any "
            "generalisation claim must rest on the FF++ cross-dataset transfer "
            "(Notebook 04), not a held-out-method test within LAV-DF. State this "
            "limitation explicitly in the paper.")
    else:
        errs.append(f"Only {n_methods} generation method(s) found among fakes. You need "
                    "at least 2 to hold one out. A method-disjoint protocol is not "
                    "possible with this data -- you would have to drop that claim from "
                    "the paper.")

if DATASET_KIND == "lavdf":
    warns.append(
        "LAV-DF 'subjects' are a SURROGATE (grouped by the fake's source video via "
        "the 'original' field), not true speaker identity -- the metadata has no "
        "speaker ID. This prevents a fake and its own real source from splitting "
        "across train/test, but the SAME real person could still appear as the "
        "source of a different 'original' video on the other side of the split. "
        "State this as a limitation, don't call it speaker-disjoint.")

if errs:
    print("!!! AUDIT FAILED !!!\n")
    for e in errs:
        print(" *", e, "\n")
    raise AssertionError("Fix parse_path (or acknowledge the data limitation) "
                         "before proceeding. Do NOT delete this check.")
else:
    print("AUDIT PASSED.")
    print("  - every file has a label")
    print(f"  - {len(subs)} subjects parseable -> subject-disjoint split possible")
    print(f"  - {n_methods} generation methods parseable -> method-disjoint split possible")
    if warns:
        print("\n  Acknowledged limitations (not failures -- but must be stated honestly")
        print("  in the paper):")
        for w in warns:
            print("   *", w, "\n")


### If the audit failed

Three possibilities, in order of likelihood:

1. **Your paths differ from the official layout.** Print a few paths, see where the
   subject / method / category actually live, and rewrite `parse_path`. This is the
   normal case with a re-upload and is just a bit of string work.

2. **The Kaggle copy is a subset** with too few subjects or only one generation
   method. Then the method-disjoint claim is not supportable on this data, and you
   must either obtain the full corpus (official request form) or **drop that claim
   from the paper**. Do not keep the claim and use a random split — that is the exact
   failure mode the paper criticises.

3. **The metadata lives in a CSV, not the paths.** Some mirrors ship a `meta_data.csv`.
   If you have one, load it and join on filename instead of regexing paths. Check:

In [ ]:
csvs = list(RAW_ROOT.rglob("*.csv"))
print("CSV files found:", csvs if csvs else "none")
if csvs:
    import pandas as pd
    dfm = pd.read_csv(csvs[0])
    print("\ncolumns:", list(dfm.columns))
    print(dfm.head())
    print("\nIf this CSV carries subject/method/category, prefer joining on it "
          "over regexing paths -- it is authoritative.")

## 1.6 Save the manifest for downstream notebooks

In [ ]:
import json

# drop rows with no parseable category -- no ground truth, nothing to train/eval on
clean_rows = [r for r in rows if r["category"] is not None]
n_dropped = len(rows) - len(clean_rows)
if n_dropped:
    print(f"Dropping {n_dropped} unparseable rows from the manifest "
          f"({n_dropped/len(rows)*100:.2f}% of {len(rows)}).")

manifest = {
    "raw_root": str(RAW_ROOT),
    "dataset_kind": DATASET_KIND,
    "feat_root": str(FEAT_ROOT),
    "runs_root": str(RUNS_ROOT),
    "n_videos": len(clean_rows),
    "n_subjects": len(subs),
    "methods": sorted({r["method"] for r in fake_rows if r["method"]}),
    "rows": clean_rows,
}
out = FEAT_ROOT / "manifest.json"
out.write_text(json.dumps(manifest, indent=1))
print(f"Wrote {out}")
print(f"  {len(clean_rows)} videos, {len(subs)} subjects, methods={manifest['methods']}")
print("\nNext: 02_extract_features.ipynb")